In [1]:
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv

load_dotenv()

loader = TextLoader("data/news.txt", encoding="utf-8")
docs = loader.load()
print(f"총 글자수: {len(docs[0].page_content)}")
print("\n========앞부분 미리보기=========\n")
print(docs[0].page_content[:500])

총 글자수: 1708

========앞부분 미리보기=========

제목: 
AI2, 상업 활용까지 자유로운 '진짜' 오픈 소스 LLM '올모' 출시

내용:
앨런AI연구소(AI2)가 완전한 오픈 소스 대형언어모델(LLM) '올모(OLMo)’를 출시했다. 데이터 수집, 학습, 배포의 전 과정을 투명하게 공개한 데다 상업적 사용까지 허용한 진정한 의미의 오픈 소스 LLM이라는 평가다.
벤처비트는 1일(현지시간) 비영리 민간 AI 연구기관인 AI2가 ‘최초의 진정한 오픈 소스 LLM 및 프레임워크’라고 소개한 ‘올모’를 출시했다고 보도했다. 
이에 따르면 올모는 모델 코드와 모델 가중치뿐만 아니라 훈련 코드, 훈련 데이터, 관련 툴킷 및 평가 툴킷도 제공한다. 이를 통해 모델이 어떻게 구축되었는지 심층적으로 분석, LLM의 작동 방식과 응답을 생성하는 원리를 더 잘 이해할 수 있다. 
올모 프레임워크는 70억 매개변수의 ‘올모 7B’ 등 4가지 변형 모델과 10억 매개변수의 ‘올모 1B’ 모델을 제공한다. 모델들은 훈련 데이터를 생성하는 코드를 포함해 


In [2]:
from langsmith import Client

prompt = Client().pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    dangerously_pull_public_prompt=True,
)
prompt.pretty_print()


Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



In [3]:
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_teddynote.callbacks import StreamingCallback

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    streaming=True,
    callbacks=[StreamingCallback()],
)

stuff_chain = create_stuff_documents_chain(llm, prompt)
answer = stuff_chain.invoke({"context": docs})

- 🚀 AI2가 데이터 수집부터 학습·배포까지 전 과정을 공개한 완전한 오픈 소스 LLM ‘OLMo’를 출시했다.
- 🔓 OLMo는 모델 코드와 가중치뿐 아니라 훈련 코드·데이터·관련 툴킷·평가 툴킷까지 제공해 LLM의 작동 원리를 분석할 수 있다.
- 🧠 70억 매개변수의 OLMo 7B 등 4가지 변형 모델과 10억 매개변수의 OLMo 1B를 제공하며, 3조 토큰 규모의 Dolma 데이터 세트를 기반으로 구축됐다.
- 📊 추론 코드, 훈련 지표·로그, 전체 모델 가중치와 단계별 체크포인트를 포함해 연구 재현성과 투명성을 높였다.
- 💼 Apache 2.0 라이선스를 적용해 상업적 활용에 제한이 없다.
- 🏆 일부 자연어 처리 벤치마크에서 Meta의 Llama와 TII의 Falcon 등 상업용 모델보다 뛰어난 성능을 보였다.
- ⚠️ 비영어권 언어 처리 품질과 코드 생성 기능은 상대적으로 부족하다.
- 🌐 OLMo의 모든 리소스는 GitHub와 Hugging Face에서 무료로 이용할 수 있다.

In [4]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """ Please summarize the sentence according to the following REQUEST.
    REQUEST:
    ...
    CONTEXT:
    {context}

    SUMMARY:
    """
)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
docs = docs[3:8]
print(f"총 페이지수: {len(docs)}")

총 페이지수: 5


In [6]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0, model="gpt-5.6-luna",)

client = Client()
map_prompt = client.pull_prompt("teddynote/map-prompt", dangerously_pull_public_prompt=True)

map_prompt.pretty_print()

================================ System Message ================================

You are a professional main thesis extractor.

================================ Human Message =================================

Your task is to extract main thesis from given documents. Answer should be in same language as given document. 

#Format: 
- thesis 1
- thesis 2
- thesis 3
- ...

Here is a given document: 
{doc}

Write 1~5 sentences.
#Answer:


In [7]:
map_chain = map_prompt | llm | StrOutputParser()

In [8]:
doc_summaries = map_chain.batch(docs)

In [9]:
print(doc_summaries[0])

- 미국 바이든 대통령은 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발과 사용을 보장하기 위한 행정명령을 발표했다.
- 행정명령은 AI 안전·보안 기준 마련, 개인정보보호, 형평성과 시민권 향상, 소비자 및 노동자 보호, 혁신과 경쟁 촉진, 국제협력을 포괄한다.
- 고성능 AI 개발 기업에는 안전성 평가 결과와 주요 정보를 정부와 공유하도록 요구하고, AI 생성 콘텐츠 식별 표준과 안전성 검증 체계 마련을 추진한다.
- 또한 법률·주택·보건 분야의 AI 차별을 방지하고, AI 연구 인프라와 전문인력 확보를 지원해 미국의 AI 경쟁력과 책임 있는 활용을 강화한다.


In [10]:
# reduce prompt 다운로드
reduce_prompt = client.pull_prompt("teddynote/reduce-prompt", dangerously_pull_public_prompt=True)

# 프롬프트 출력
reduce_prompt.pretty_print()

================================ System Message ================================

You are a professional summarizer. You are given a list of summaries of documents and you are asked to create a single summary of the documents.

================================ Human Message =================================

#Instructions: 
1. Extract main points from a list of summaries of documents
2. Make final summaries in bullet points format.
3. Answer should be written in {language}.

#Format: 
- summary 1
- summary 2
- summary 3
- ...

Here is a list of summaries of documents: 
{doc_summaries}

#SUMMARY:


In [11]:
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [12]:
from langchain_teddynote.messages import stream_response

answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

stream_response(answer)

d:\rag_two_branch\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


- 미국 바이든 행정부는 2023년 10월 안전하고 신뢰할 수 있는 AI의 개발·활용을 위한 행정명령을 발표하고, AI 안전·보안 기준, 개인정보보호, 차별 방지, 소비자·노동자 보호, 혁신·경쟁 촉진 및 국제협력을 종합적으로 추진하기로 했다.
- 고성능 AI 개발 기업에는 안전성 평가 결과와 주요 정보를 정부에 공유하도록 요구하고, AI 생성 콘텐츠 식별·출처 확인 표준과 안전성 검증 체계를 마련하는 한편, 법률·주택·보건 분야에서의 AI 차별을 방지하고 연구 인프라와 전문인력 확보를 지원한다.
- G7 히로시마 AI 프로세스는 기반모델과 생성 AI의 위험을 AI 수명주기 전반에서 평가·완화하기 위한 국제 행동강령을 마련했다. 행동강령은 투명성·책임성, 사고 및 정보 공유, 개인정보보호·보안 강화, 워터마크 등 콘텐츠 인증을 권고하며, 기업의 자발적 채택과 기술 발전에 따른 지속적인 개정을 전제로 한다.
- AI 안전성 정상회의 참가 28개국은 블레츨리 선언을 통해 정부·국제기구·기업·학계·시민사회가 첨단 AI 위험에 공동 대응하기로 했다. 각국은 안전 테스트 도구와 평가 지표, 공동 표준 및 연구 역량을 강화하고 테스트 결과를 공유하기로 했으며, 한국은 영국·프랑스와 후속 AI 정상회의를 공동 개최할 예정이다.
- 미국 법원은 예술가들이 미드저니·스태빌리티AI·디비언트아트를 상대로 제기한 생성 AI 저작권 침해 소송 일부를 기각했다. 저작권 미등록과 생성 이미지와 원작 간 실질적 유사성 입증 부족이 주요 사유였지만, 등록 작품 16개의 무단 복제와 관련된 스태빌리티AI 소송은 계속 진행된다.
- 미국 FTC는 생성 AI가 개인정보 침해, 차별·편견의 자동화, 사기와 소비자 기만, 창작자의 경쟁력·평판·저작물 가치 훼손을 초래할 수 있다고 경고했다. 또한 빅테크의 독점적 데이터와 자원을 활용한 시장 지배력 강화를 우려하며, 저작권뿐 아니라 소비자 보호와 공정경쟁 차원에서 규제하겠다고 밝혔다.

In [13]:
from langchain_core.runnables import chain

@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0, model="gpt-5.6-luna",
    )

    # map_prompt 다운로드
    map_prompt = client.pull_prompt("teddynote/map-prompt", dangerously_pull_public_prompt=True)

    # map chain 생성
    map_chain = map_prompt | map_llm | StrOutputParser()

    # 첫 번째 프롬프트, ChatOpenAI, 문자열 출력 파서를 연결하여 체인을 생성
    doc_summaries = map_chain.batch(docs)

    # reduce_prompt 다운로드
    reduce_prompt = client.pull_prompt("teddynote/reduce-prompt", dangerously_pull_public_prompt=True)
    reduce_llm = ChatOpenAI(
        model="gpt-5.6-luna",
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

In [14]:
answer = map_reduce_chain.invoke(docs)

- 미국은 2023년 10월 행정명령을 통해 안전·보안 기준 마련, 개인정보 보호, 차별 방지, 소비자·노동자 보호, 혁신·경쟁 촉진 및 국제협력을 포함한 포괄적인 AI 규제·지원 체계를 추진했다.
- 고성능 AI 개발 기업에는 정보 공유와 안전성 검증을 요구하고, 정부는 연구 인프라 확충과 전문 인력 유입을 지원해 AI의 안전한 개발과 활용을 강화할 계획이다.
- G7 히로시마 AI 프로세스는 AI 수명주기 전반의 위험 관리, 투명성·책임성, 보안 통제, 콘텐츠 출처 확인, 이해관계자 간 정보 공유를 포함한 자발적 국제 행동강령을 마련했다.
- 28개국은 블레츨리 선언을 통해 국가·국제기구·기업·학계·시민사회의 협력과 첨단 AI 기업의 안전 책임을 강조하고, 공동 안전 평가·테스트 도구·표준 개발 및 결과 공유에 합의했다.
- 영국은 AI 안전 연구소를 설립해 첨단 AI 안전성 평가를 주도하며, 참가국들은 AI 위험과 가능성을 분석하는 과학 보고서 작성 및 후속 국제회의 개최를 추진하기로 했다.
- 미국 법원은 생성형 AI 저작권 소송 대부분을 저작권 등록 및 증거 부족으로 기각했지만, 특정 작품의 무단 복제 혐의와 관련된 소송은 계속 허용해 AI 학습 데이터와 저작권 책임 문제가 여전히 쟁점임을 보여줬다.
- 미국 FTC는 생성형 AI가 개인정보 침해, 차별, 사기, 창작자 작품·정체성 모방 및 시장 독점 등을 초래할 수 있다고 경고하고, 기존 법 집행 권한을 활용해 소비자 보호와 공정한 경쟁을 보장하겠다고 밝혔다.

In [15]:
from dotenv import load_dotenv
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

map_llm = ChatOpenAI(
    model="gpt-5.6-luna",
)

# map chain 생성
client = Client()
map_summary = client.pull_prompt("teddynote/map-summary-prompt", dangerously_pull_public_prompt=True)

map_summary.pretty_print()

================================ System Message ================================

You are an expert summarizer. Your task is to summarize the following document in {language}.

================================ Human Message =================================

Extract most important main thesis from the documents, then summarize in bullet points.

#Format:
- summary 1
- summary 2
- summary 3
-...

Here is a given document: 
{documents}

Write 1~5 sentences. Think step by step.
#Summary:


In [16]:
# map chain 생성
map_chain = map_summary | map_llm | StrOutputParser()

In [17]:
# 첫 번째 문서의 요약 출력
print(map_chain.invoke({"documents": docs[0], "language": "Korean"}))

- 미국 바이든 대통령은 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발·활용을 위해 연방 차원의 행정명령을 발표했다.
- 행정명령은 고성능 AI 개발기업의 안전성 평가 및 정보 공유, AI 생성물 표시 기준 마련, 개인정보 보호와 차별 방지 등 규제·책임 강화를 핵심으로 한다.
- 또한 소비자·근로자 보호, AI 연구 인프라 확대, 중소기업 지원, 해외 인재 유치 및 국제협력을 통해 AI 혁신과 경쟁력도 함께 촉진하고자 한다.


In [18]:
input_doc = [{"documents": doc, "language": "Korean"} for doc in docs]
input_doc

[{'documents': Document(metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 3, 'page_label': '4'}, page_content='1. 정책/법제  2. 기업/산업 3. 기술/연구  4. 인력/교육\n미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표 \nn 미국 바이든 대통령이 ‘안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령’에 서명하고 \n광범위한 행정 조치를 명시\nn 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자 \n보호 △노동자 지원 △혁신과 경쟁 촉진 △국제협력을 골자로 함\nKEY Contents\n£ 바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진\nn 미국 바이든 대통령이 2023년 10월 30일 연방정부 차원에서 안전하고 신뢰할 수 있는 AI 개발과 \n사용을 보장하기 위한 행정명령을 발표\n∙ 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자 보호 \n△노동자 지원 △혁신과 경쟁 촉진 △국제협력에 관한 내용을 포괄\nn (AI 안전과 보안 기준) 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 \n주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 \nAI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진\n∙ △1026 플롭스(FLOPS, Floa

In [19]:
print(map_chain.batch(input_doc))

['#Summary:\n- 미국 바이든 대통령은 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발·활용을 위한 행정명령을 발표했다.\n- 행정명령은 고성능 AI 개발 기업의 안전성 평가 결과와 주요 정보를 정부에 공유하도록 하고, AI 안전 기준 및 생성 콘텐츠 표시 표준 마련을 추진한다.\n- 개인정보보호, 차별·편견 방지, 소비자·노동자 보호 등 AI 활용에 따른 사회적 위험을 줄이기 위한 지침과 정책을 강화한다.\n- 국가 AI 연구 인프라 확대, 중소기업·개발자 지원, 해외 AI 인재 유치 등을 통해 혁신과 경쟁을 촉진하고 국제협력을 강화한다.', '- G7은 첨단 AI 시스템의 위험을 식별·완화하기 위해 기업의 자발적 채택을 권고하는 국제 행동강령에 합의했다.\n- 행동강령은 AI 수명주기 전반의 위험 관리, 투명성·책임성 강화, 사고 및 정보 공유, 개인정보·지식재산권 보호를 요구한다.\n- 또한 물리·사이버·내부자 보안 통제와 워터마크 등 AI 생성 콘텐츠의 인증·출처 확인 체계 구축을 제시한다.\n- G7은 사회적 위험 완화와 기후·보건·교육 등 글로벌 문제 해결을 위한 AI 연구 및 국제 표준 개발을 우선 추진할 계획이다.', '- 영국 AI 안전성 정상회의에 참가한 28개국은 첨단 AI의 위험에 공동 대응하기 위한 ‘블레츨리 선언’을 채택하고, 국가·기업·학계·시민사회 간 협력을 강화하기로 했다.\n- 첨단 AI 개발 기업의 투명성 제고, 안전성 평가 지표와 테스트 도구 개발, 정부 주도의 외부 안전성 시험 및 결과 공유에 합의했다.\n- 영국 AI 안전 연구소가 국제 협력과 첨단 AI 안전 테스트를 주도하며, 참가국들은 AI 위험·가능성에 대한 과학적 평가와 공동 표준 마련도 추진할 예정이다.\n- 한국은 영국과 6개월 후 온라인 AI 미니 정상회의를, 프랑스와는 1년 후 대면 정상회의를 공동 개최하기로 했다.', '- 미국 캘리포니아 북부지방법원은 예술가 3인이 미드저니·스태빌리티AI·디비언트아트를 상대로 제기한 생성 AI 저

In [20]:
refine_prompt = client.pull_prompt("teddynote/refine-prompt", dangerously_pull_public_prompt=True)

refine_prompt.pretty_print()

================================ System Message ================================

You are an expert summarizer.

================================ Human Message =================================

Your job is to produce a final summary

We have provided an existing summary up to a certain point:
{previous_summary}

We have the opportunity to refine the existing summary(only if needed) with some more context below.
------------
{current_summary}
------------
Given the new context, refine the original summary in {language}.
If the context isn't useful, return the original summary.


In [21]:
refine_llm = ChatOpenAI(
    model="gpt-5.6-luna",
)

# refine_chain 생성
refine_chain = refine_prompt | refine_llm | StrOutputParser()

In [24]:
from langchain_core.runnables import chain

@chain
def map_refine_chain(docs):

    # map chain 생성
    map_summary = client.pull_prompt("teddynote/map-summary-prompt", dangerously_pull_public_prompt=True)

    map_chain = (
        map_summary
        | ChatOpenAI(
            model="gpt-5.6-luna",
        )
        | StrOutputParser()
    )

    input_doc = [{"documents": doc.page_content, "language": "Korean"} for doc in docs]

    # 첫 번째 프롬프트, ChatOpenAI, 문자열 출력 파서를 연결하여 체인을 생성
    doc_summaries = map_chain.batch(input_doc)
    refine_prompt = client.pull_prompt("teddynote/refine-prompt", dangerously_pull_public_prompt=True)

    refine_llm = ChatOpenAI(
        model = "gpt-5.6-luna",
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    refine_chain = refine_prompt | refine_llm | StrOutputParser()

    previous_summary = doc_summaries[0]

    for current_summary in doc_summaries[1:]:

        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )
        print("\n\n----------------\n\n")

    return previous_summary



In [25]:
refine_summary = map_refine_chain.invoke(docs)

- 바이든 행정부는 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발·사용을 위해 연방 차원의 포괄적 행정명령을 발표했다.
- 행정명령은 고성능 AI의 안전성 테스트와 정보 공유, AI 생성 콘텐츠의 표시·인증 기준 마련, 개인정보 보호 및 보안 강화를 추진한다.
- 법률·주택·보건 분야의 알고리즘 편견과 차별을 방지하고, 소비자·근로자 보호와 AI 교육 지원을 확대한다.
- 국가 AI 연구자원과 중소기업 지원을 통해 혁신과 경쟁을 촉진하는 한편, 국제협력과 해외 AI 인재 유입도 강화한다.
- 이와 함께 G7은 히로시마 AI 프로세스를 통해 첨단·생성 AI 기업이 AI 수명주기 전반의 위험을 식별·완화하도록 하는 국제 행동강령에 합의했다.
- 행동강령은 AI의 성능과 한계, 사고 및 관련 정보 공개·공유, 책임 있는 거버넌스, 물리·사이버·내부자 보안 강화를 요구하며, 워터마크 등 생성 콘텐츠의 인증·출처 확인 체계 마련도 강조한다.
- 또한 개인정보와 지식재산권 보호, 국제 표준 개발을 추진하고, 사회적 위험을 줄이는 동시에 기후·보건·교육 등 글로벌 문제 해결에 기여하는 AI 개발을 우선시한다.

----------------


- 바이든 행정부는 2023년 10월 30일 안전하고 신뢰할 수 있는 AI의 개발·사용을 위해 연방 차원의 포괄적 행정명령을 발표했다.
- 행정명령은 고성능 AI의 안전성 테스트와 정보 공유, AI 생성 콘텐츠의 표시·인증 기준 마련, 개인정보 보호 및 보안 강화를 추진한다.
- 법률·주택·보건 분야의 알고리즘 편견과 차별을 방지하고, 소비자·근로자 보호와 AI 교육 지원을 확대한다.
- 국가 AI 연구자원과 중소기업 지원을 통해 혁신과 경쟁을 촉진하는 한편, 국제협력과 해외 AI 인재 유입도 강화한다.
- G7은 히로시마 AI 프로세스를 통해 첨단·생성 AI 기업이 AI 수명주기 전반의 위험을 식별·완화하도록 하는 국제 행동강령에 합의했다. 행동강령은 AI의 성능과 한계, 사고 관련 정보의 공개·공유, 책임 있는 